# 05 — Validation

Cross-validate the specificity results against known markers and
external databases:

1. **Known marker genes** — Do we recover established cell type markers?
2. **Human Protein Atlas (HPA)** — Does protein-level data agree?
3. **Sensitivity analysis** — How do results change with thresholds?

**Inputs:** Specificity results + preprocessed data  
**Outputs:** Validation report

In [ ]:
import sys
from pathlib import Path
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(name)s | %(message)s")

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

## 1. Load Data

In [ ]:
mean_cpm = pd.read_parquet(DATA_DIR / "preprocessed_mean_cpm.parquet")
det_df = pd.read_parquet(DATA_DIR / "preprocessed_detection_rate.parquet")
results = pd.read_csv(RESULTS_DIR / "specificity_scores.csv")
pass_only = results[results["all_pass"]]

print(f"Results: {len(results):,} total, {len(pass_only):,} passing all thresholds")

## 2. Known Marker Gene Validation

Check whether established cell type markers are recovered by our pipeline.

In [ ]:
# Well-established cell type marker genes
KNOWN_MARKERS = {
    # Gene: expected primary cell type (case-insensitive partial match)
    "ALB": "hepatocyte",
    "APOB": "hepatocyte",
    "CYP3A4": "hepatocyte",
    "CD3D": "t cell",
    "CD3E": "t cell",
    "CD8A": "t cell",  # CD8+ T cells
    "CD79A": "b cell",
    "MS4A1": "b cell",  # CD20
    "CD14": "monocyte",
    "FCGR3A": "monocyte",  # CD16, non-classical monocyte or NK
    "NKG7": "nk cell",
    "GNLY": "nk cell",
    "S100A8": "neutrophil",
    "PECAM1": "endothelial",  # CD31
    "VWF": "endothelial",
    "COL1A1": "fibroblast",
    "DCN": "fibroblast",
    "TNNT2": "cardiomyocyte",
    "MYH7": "cardiomyocyte",
    "SNAP25": "neuron",
    "SYP": "neuron",
    "KRT14": "keratinocyte",
    "JCHAIN": "plasma cell",
    "MZB1": "plasma cell",
    "SFTPC": "type ii pneumocyte",  # alveolar type II
    "AQP2": "collecting duct",      # kidney collecting duct
    "INS": "beta cell",             # pancreatic beta cell
}

print(f"{'Gene':<10} {'Expected':<25} {'Found Cell Type':<30} {'Tau':>6} {'FC':>8} {'Pass?':>6}")
print("=" * 90)

recovered = 0
tested = 0

for gene, expected in KNOWN_MARKERS.items():
    if gene not in mean_cpm.index:
        print(f"{gene:<10} {expected:<25} {'NOT IN DATA':<30}")
        continue
    
    tested += 1
    gene_results = results[results["gene"] == gene]
    
    if gene_results.empty:
        top_ct = mean_cpm.loc[gene].idxmax()
        print(f"{gene:<10} {expected:<25} {top_ct:<30} {'<0.85':>6} {'N/A':>8} {'NO':>6}")
    else:
        row = gene_results.iloc[0]
        found_ct = row["cell_type"]
        match = expected.lower() in found_ct.lower()
        if match:
            recovered += 1
        print(
            f"{gene:<10} {expected:<25} {found_ct:<30} "
            f"{row['tau']:>6.3f} {row['fold_change']:>8.1f} "
            f"{'YES' if match else 'CHECK':>6}"
        )

print(f"\nRecovery rate: {recovered}/{tested} ({recovered/max(tested,1)*100:.0f}%)")

## 3. Threshold Sensitivity Analysis

How does the number of hits change as we vary each threshold?

In [ ]:
from src.specificity import score_specificity

# Vary Tau threshold
tau_values = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.98]
tau_hits = []
for tau_t in tau_values:
    r = score_specificity(mean_cpm, det_df, tau_threshold=tau_t)
    n_pass = r["all_pass"].sum() if not r.empty else 0
    tau_hits.append(n_pass)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(tau_values, tau_hits, "bo-")
axes[0].axvline(0.85, color="red", linestyle="--", alpha=0.5, label="Current threshold")
axes[0].set_xlabel("Tau Threshold")
axes[0].set_ylabel("Genes Passing All Thresholds")
axes[0].set_title("Sensitivity to Tau Threshold")
axes[0].legend()

# Vary fold-change threshold
fc_values = [2, 5, 10, 20, 50, 100]
fc_hits = []
for fc_t in fc_values:
    r = score_specificity(mean_cpm, det_df, fc_threshold=fc_t)
    n_pass = r["all_pass"].sum() if not r.empty else 0
    fc_hits.append(n_pass)

axes[1].plot(fc_values, fc_hits, "ro-")
axes[1].axvline(10, color="blue", linestyle="--", alpha=0.5, label="Current threshold")
axes[1].set_xlabel("Fold-Change Threshold")
axes[1].set_ylabel("Genes Passing All Thresholds")
axes[1].set_title("Sensitivity to Fold-Change Threshold")
axes[1].set_xscale("log")
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "threshold_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Cross-Reference with Human Protein Atlas (Optional)

If you have HPA data downloaded, this section validates at the protein level.
Download from: https://www.proteinatlas.org/about/download (normal_tissue.tsv.zip)

In [ ]:
HPA_PATH = PROJECT_ROOT / "data" / "references" / "normal_tissue.tsv"

if HPA_PATH.exists():
    hpa = pd.read_csv(HPA_PATH, sep="\t")
    print(f"HPA data loaded: {len(hpa):,} rows")
    print(f"Columns: {list(hpa.columns)}")
    
    # Check top hits against HPA
    top_genes = pass_only.head(20)["gene"].tolist()
    hpa_matches = hpa[hpa["Gene name"].isin(top_genes)]
    if not hpa_matches.empty:
        print(f"\nHPA entries for top 20 genes: {len(hpa_matches)}")
        display(hpa_matches.head(20))
else:
    print("HPA data not found. Download from https://www.proteinatlas.org/about/download")
    print("Place normal_tissue.tsv in data/references/")

## 5. Validation Summary

In [ ]:
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"Total genes analysed:        {mean_cpm.shape[0]:>8,}")
print(f"Genes with Tau >= 0.85:      {len(results):>8,}")
print(f"Passing all thresholds:      {len(pass_only):>8,}")
print(f"Cell types with hits:        {pass_only['cell_type'].nunique():>8}")
print(f"Known markers tested:        {tested:>8}")
print(f"Known markers recovered:     {recovered:>8} ({recovered/max(tested,1)*100:.0f}%)")
print(f"\nThresholds used:")
print(f"  Tau:                 >= 0.85")
print(f"  On-target CPM:       >= 1.0")
print(f"  On-target detection: >= 25%")
print(f"  Off-target detection:<= 5%")
print(f"  Fold-change:         >= 10x")

## Summary

Validation complete. Results are ready for use in the **Streamlit query interface** (`app/streamlit_app.py`).